# Fine-Tune VinDr Models on INbreast

**Transfer learning: Fine-tune optimized models on the target dataset**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dtobi59/mammography-multiobjective-optimization/blob/main/finetune_inbreast.ipynb)

This notebook:
1. Loads Pareto-optimal models trained on VinDr-Mammo
2. Fine-tunes them on INbreast dataset
3. Evaluates both zero-shot and fine-tuned performance
4. Compares transfer learning effectiveness
5. Saves fine-tuned models and results

**Prerequisites:**
- Completed NSGA-III optimization (colab_tutorial.ipynb)
- VinDr model checkpoints saved in Google Drive
- INbreast dataset prepared (PNG format)

**Author:** David ([@dtobi59](https://github.com/dtobi59))

## 1. Setup Environment

Check GPU and clone repository.

In [ ]:
# Check GPU availability
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Clone the repository
!git clone https://github.com/dtobi59/mammography-multiobjective-optimization.git

# Change to project directory
%cd mammography-multiobjective-optimization

# List files
!ls -la

In [ ]:
# Install required packages
!pip install -q -r requirements.txt

print("\n[SUCCESS] All dependencies installed!")

In [ ]:
# Setup Python path
import sys
import os

project_root = os.getcwd()
print(f"Project root: {project_root}")

if project_root not in sys.path:
    sys.path.insert(0, project_root)
    print(f"Added {project_root} to sys.path")

print(f"\nPython sys.path[0]: {sys.path[0]}")
print("[OK] Path setup complete!")

## 2. Mount Google Drive

Access your VinDr models and INbreast dataset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Set paths to your data in Google Drive
INBREAST_PATH = "/content/drive/MyDrive/INbreast"
OPTIMIZATION_DIR = "/content/drive/MyDrive/vindr_optimization"
CHECKPOINT_DIR = f"{OPTIMIZATION_DIR}/checkpoints"
RESULTS_DIR = f"{OPTIMIZATION_DIR}/results"

# Directory for fine-tuned models
FINETUNE_DIR = f"{OPTIMIZATION_DIR}/inbreast_finetuned"

print("\n[SUCCESS] Google Drive mounted!")
print(f"INbreast dataset: {INBREAST_PATH}")
print(f"VinDr models: {CHECKPOINT_DIR}")
print(f"Fine-tuned models will be saved to: {FINETUNE_DIR}")

## 3. Load VinDr Optimization Results

Load the Pareto front to select a model for fine-tuning.

In [ ]:
import glob
import pandas as pd
from pathlib import Path

# Find Pareto solutions CSV files
pareto_files = sorted(glob.glob(f"{RESULTS_DIR}/pareto_solutions_*.csv"))

print("=" * 80)
print("AVAILABLE VINDR MODELS")
print("=" * 80)

if not pareto_files:
    print("[ERROR] No Pareto solutions found!")
    print(f"\nExpected location: {RESULTS_DIR}")
    print("\nPlease run the optimization notebook first (colab_tutorial.ipynb)")
else:
    print(f"Found {len(pareto_files)} result file(s):\n")
    for i, file in enumerate(pareto_files):
        filename = Path(file).name
        print(f"  {i+1}. {filename}")
    print()
    
    # Load the most recent results
    latest_results = pareto_files[-1]
    print(f"Loading most recent: {Path(latest_results).name}")
    
    pareto_df = pd.read_csv(latest_results)
    print(f"\n[OK] Loaded {len(pareto_df)} Pareto-optimal solutions")
    print("=" * 80)

In [ ]:
# Display Pareto front summary
print("=" * 80)
print("PARETO FRONT SUMMARY")
print("=" * 80)
print(f"Total solutions: {len(pareto_df)}\n")

print("Best solutions for each objective:\n")

best_pr_auc_idx = pareto_df['pr_auc'].idxmax()
print(f"  Best PR-AUC:")
print(f"    Solution ID: {best_pr_auc_idx}")
print(f"    PR-AUC:  {pareto_df.loc[best_pr_auc_idx, 'pr_auc']:.4f}")
print(f"    AUROC:   {pareto_df.loc[best_pr_auc_idx, 'auroc']:.4f}")
print(f"    Brier:   {pareto_df.loc[best_pr_auc_idx, 'brier']:.4f}")
print()

best_auroc_idx = pareto_df['auroc'].idxmax()
print(f"  Best AUROC:")
print(f"    Solution ID: {best_auroc_idx}")
print(f"    PR-AUC:  {pareto_df.loc[best_auroc_idx, 'pr_auc']:.4f}")
print(f"    AUROC:   {pareto_df.loc[best_auroc_idx, 'auroc']:.4f}")
print(f"    Brier:   {pareto_df.loc[best_auroc_idx, 'brier']:.4f}")
print()

print("=" * 80)

# Display first few solutions
print("\nFirst 10 solutions:")
pareto_df.head(10)

## 4. Load INbreast Dataset

Load and prepare the INbreast dataset for fine-tuning.

**Note:** Make sure you've converted DICOM files to PNG (see zero_shot_evaluation.ipynb)

In [ ]:
# Update config with INbreast path
import os
with open('config.py', 'r') as f:
    config_content = f.read()

# Update INbreast path
config_content = config_content.replace(
    'INBREAST_PATH = "/content/drive/MyDrive/INbreast"',
    f'INBREAST_PATH = "{INBREAST_PATH}"'
)

# Use PNG images
if '"image_dir": "images"' in config_content:
    config_content = config_content.replace(
        '"image_dir": "images"',
        '"image_dir": "images_png"'
    )
elif '"image_dir": "AllDICOMs"' in config_content:
    config_content = config_content.replace(
        '"image_dir": "AllDICOMs"',
        '"image_dir": "images_png"'
    )

with open('config.py', 'w') as f:
    f.write(config_content)

# Reload config module
import importlib
if 'config' in sys.modules:
    importlib.reload(config)
else:
    import config

print("[OK] Configuration updated!")
print(f"INbreast path: {INBREAST_PATH}")
print(f"Image directory: {config.INBREAST_CONFIG['image_dir']}")

In [ ]:
import config
from optimization.nsga3_runner import load_metadata
from data.dataset import create_train_val_split

print("=" * 80)
print("LOADING INBREAST DATASET")
print("=" * 80)

# Load INbreast metadata
inbreast_metadata = load_metadata(
    dataset_name="inbreast",
    dataset_path=config.INBREAST_PATH,
    dataset_config=config.INBREAST_CONFIG
)

print(f"\n[OK] Loaded {len(inbreast_metadata)} images")
print(f"Patients: {inbreast_metadata['patient_id'].nunique()}")
print(f"Breasts: {inbreast_metadata['breast_id'].nunique()}")
print()
print("Label distribution:")
print(inbreast_metadata['label'].value_counts())
print()
print("View distribution:")
print(inbreast_metadata['view'].value_counts())

# Create train/val split (patient-wise to prevent leakage)
print("\n" + "=" * 80)
print("CREATING TRAIN/VAL SPLIT (PATIENT-WISE)")
print("=" * 80)

inbreast_train, inbreast_val = create_train_val_split(
    inbreast_metadata,
    train_ratio=0.8,
    random_seed=config.RANDOM_SEED
)

print(f"\nTrain: {len(inbreast_train)} images")
print(f"  Malignant: {(inbreast_train['label'] == 1).sum()}")
print(f"  Benign: {(inbreast_train['label'] == 0).sum()}")
print(f"  Patients: {inbreast_train['patient_id'].nunique()}")

print(f"\nVal: {len(inbreast_val)} images")
print(f"  Malignant: {(inbreast_val['label'] == 1).sum()}")
print(f"  Benign: {(inbreast_val['label'] == 0).sum()}")
print(f"  Patients: {inbreast_val['patient_id'].nunique()}")

print("=" * 80)

## 5. Select Model for Fine-Tuning

Choose which VinDr model to fine-tune on INbreast.

In [ ]:
# ============================================================================
# SELECT WHICH MODEL TO FINE-TUNE
# ============================================================================
# Change this to fine-tune different solutions:

solution_id = pareto_df['pr_auc'].idxmax()  # Best PR-AUC (default)
# solution_id = pareto_df['auroc'].idxmax()    # Best AUROC
# solution_id = pareto_df['brier'].idxmin()    # Best Brier
# solution_id = 5                               # Specific solution ID

# ============================================================================

selected_solution = pareto_df.iloc[solution_id]

print("=" * 80)
print(f"SELECTED MODEL: SOLUTION {solution_id}")
print("=" * 80)
print()
print("Hyperparameters:")
print(f"  Learning rate:          {selected_solution['learning_rate']:.6f}")
print(f"  Weight decay:           {selected_solution['weight_decay']:.6f}")
print(f"  Dropout rate:           {selected_solution['dropout_rate']:.4f}")
print(f"  Augmentation strength:  {selected_solution['augmentation_strength']:.4f}")
print(f"  Unfreeze fraction:      {selected_solution['unfreeze_fraction']:.4f}")
print()
print("VinDr (source) performance:")
print(f"  PR-AUC:     {selected_solution['pr_auc']:.4f}")
print(f"  AUROC:      {selected_solution['auroc']:.4f}")
print(f"  Brier:      {selected_solution['brier']:.4f}")
print("=" * 80)

## 6. Load VinDr Model

Load the pre-trained model checkpoint.

In [ ]:
from pathlib import Path
from models.resnet import ResNet50WithPartialFineTuning
import torch

print("=" * 80)
print("LOADING VINDR MODEL CHECKPOINT")
print("=" * 80)

# Find checkpoint
checkpoint_path = Path(CHECKPOINT_DIR) / f"eval_{solution_id}" / "best_checkpoint.pt"

if not checkpoint_path.exists():
    print(f"[ERROR] Checkpoint not found: {checkpoint_path}")
    print()
    print("Available checkpoint directories:")
    for d in sorted(Path(CHECKPOINT_DIR).glob("eval_*")):
        print(f"  {d.name}")
    raise FileNotFoundError(f"Checkpoint not found for solution {solution_id}")

print(f"Loading: {checkpoint_path}")
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
print(f"[OK] Checkpoint loaded")
print(f"  Epoch: {checkpoint.get('epoch', 'unknown')}")
print(f"  Best PR-AUC: {checkpoint.get('best_pr_auc', 'unknown')}")
print()

# Create model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = ResNet50WithPartialFineTuning(
    dropout_rate=selected_solution['dropout_rate'],
    unfreeze_fraction=selected_solution['unfreeze_fraction'],
)

model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)

print("[OK] Model created and VinDr weights loaded")
print("=" * 80)

## 7. Evaluate Zero-Shot Performance (Baseline)

First, evaluate the VinDr model on INbreast without fine-tuning.

In [ ]:
from data.dataset import MammographyDataset, get_base_transform
from training.metrics import compute_metrics
from utils.noisy_or import aggregate_to_breast_level
import numpy as np

print("=" * 80)
print("ZERO-SHOT EVALUATION (BEFORE FINE-TUNING)")
print("=" * 80)

# Create validation dataloader
inbreast_image_dir = str(Path(config.INBREAST_PATH) / config.INBREAST_CONFIG["image_dir"])
num_workers = getattr(config, 'NUM_WORKERS', 2)

base_transform = get_base_transform()
val_dataset = MammographyDataset(
    metadata=inbreast_val,
    image_dir=inbreast_image_dir,
    transform=base_transform,
    augmentation=None,
)

val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
)

# Run inference
model.eval()
all_preds = []
all_labels = []
all_image_ids = []

with torch.no_grad():
    for images, labels, image_ids in val_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        probs = model(images)
        
        all_preds.extend(probs.cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())
        all_image_ids.extend(image_ids)

# Compute metrics
all_preds_array = np.array(all_preds)
all_labels_array = np.array(all_labels)

zeroshot_image_metrics = compute_metrics(all_preds_array, all_labels_array)

# Breast-level metrics
image_predictions = {img_id: pred for img_id, pred in zip(all_image_ids, all_preds)}
breast_preds, breast_labels = aggregate_to_breast_level(
    image_predictions=image_predictions,
    metadata=inbreast_val
)
zeroshot_breast_metrics = compute_metrics(breast_preds, breast_labels)

print(f"\nValidation set: {len(all_preds)} images")
print()
print("Image-level metrics:")
print(f"  PR-AUC:  {zeroshot_image_metrics['pr_auc']:.4f}")
print(f"  AUROC:   {zeroshot_image_metrics['auroc']:.4f}")
print(f"  Brier:   {zeroshot_image_metrics['brier']:.4f}")
print()
print("Breast-level metrics:")
print(f"  PR-AUC:  {zeroshot_breast_metrics['pr_auc']:.4f}")
print(f"  AUROC:   {zeroshot_breast_metrics['auroc']:.4f}")
print(f"  Brier:   {zeroshot_breast_metrics['brier']:.4f}")
print("=" * 80)

## 8. Fine-Tune on INbreast

Fine-tune the VinDr model on INbreast training set.

In [ ]:
# ============================================================================
# FINE-TUNING CONFIGURATION
# ============================================================================

# Fine-tuning hyperparameters
FINETUNE_EPOCHS = 20  # Fewer epochs than initial training
FINETUNE_LR = selected_solution['learning_rate'] * 0.1  # Lower learning rate for fine-tuning
FINETUNE_WEIGHT_DECAY = selected_solution['weight_decay']
FINETUNE_AUG_STRENGTH = selected_solution['augmentation_strength']

# Early stopping patience
PATIENCE = 5

print("=" * 80)
print("FINE-TUNING CONFIGURATION")
print("=" * 80)
print(f"Epochs: {FINETUNE_EPOCHS}")
print(f"Learning rate: {FINETUNE_LR:.6f} (10% of original)")
print(f"Weight decay: {FINETUNE_WEIGHT_DECAY:.6f}")
print(f"Augmentation: {FINETUNE_AUG_STRENGTH:.4f}")
print(f"Early stopping patience: {PATIENCE}")
print("=" * 80)

In [ ]:
from data.dataset import create_dataloaders

# Create dataloaders
train_loader, finetune_val_loader = create_dataloaders(
    train_metadata=inbreast_train,
    val_metadata=inbreast_val,
    image_dir=inbreast_image_dir,
    batch_size=config.BATCH_SIZE,
    augmentation_strength=FINETUNE_AUG_STRENGTH,
    num_workers=num_workers,
)

print(f"[OK] Created dataloaders")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(finetune_val_loader)}")
print(f"  Batch size: {config.BATCH_SIZE}")

In [ ]:
import torch.nn as nn
from tqdm import tqdm

print("=" * 80)
print("FINE-TUNING ON INBREAST")
print("=" * 80)
print()

# Setup optimizer and loss
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=FINETUNE_LR,
    weight_decay=FINETUNE_WEIGHT_DECAY
)

criterion = nn.BCELoss()

# Training state
best_pr_auc = 0.0
patience_counter = 0
train_history = []

for epoch in range(FINETUNE_EPOCHS):
    # Training
    model.train()
    train_loss = 0.0
    train_preds = []
    train_labels = []
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{FINETUNE_EPOCHS}")
    for images, labels, _ in pbar:
        images = images.to(device)
        labels = labels.float().to(device)
        
        optimizer.zero_grad()
        outputs = model(images).squeeze()
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        train_preds.extend(outputs.detach().cpu().numpy().tolist())
        train_labels.extend(labels.cpu().numpy().tolist())
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    train_loss /= len(train_loader)
    train_metrics = compute_metrics(np.array(train_preds), np.array(train_labels))
    
    # Validation
    model.eval()
    val_loss = 0.0
    val_preds = []
    val_labels = []
    
    with torch.no_grad():
        for images, labels, _ in finetune_val_loader:
            images = images.to(device)
            labels = labels.float().to(device)
            
            outputs = model(images).squeeze()
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            val_preds.extend(outputs.cpu().numpy().tolist())
            val_labels.extend(labels.cpu().numpy().tolist())
    
    val_loss /= len(finetune_val_loader)
    val_metrics = compute_metrics(np.array(val_preds), np.array(val_labels))
    
    # Log epoch results
    print(f"\nEpoch {epoch+1}/{FINETUNE_EPOCHS}")
    print(f"  Train - Loss: {train_loss:.4f}, PR-AUC: {train_metrics['pr_auc']:.4f}, AUROC: {train_metrics['auroc']:.4f}")
    print(f"  Val   - Loss: {val_loss:.4f}, PR-AUC: {val_metrics['pr_auc']:.4f}, AUROC: {val_metrics['auroc']:.4f}")
    
    train_history.append({
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'train_pr_auc': train_metrics['pr_auc'],
        'train_auroc': train_metrics['auroc'],
        'val_loss': val_loss,
        'val_pr_auc': val_metrics['pr_auc'],
        'val_auroc': val_metrics['auroc'],
    })
    
    # Save best model
    if val_metrics['pr_auc'] > best_pr_auc:
        best_pr_auc = val_metrics['pr_auc']
        patience_counter = 0
        
        # Save checkpoint
        os.makedirs(FINETUNE_DIR, exist_ok=True)
        checkpoint_save_path = Path(FINETUNE_DIR) / f"solution_{solution_id}_finetuned.pt"
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_pr_auc': best_pr_auc,
            'val_metrics': val_metrics,
            'solution_id': solution_id,
        }, checkpoint_save_path)
        
        print(f"  ✓ New best PR-AUC: {best_pr_auc:.4f} (saved)")
    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{PATIENCE})")
        
        if patience_counter >= PATIENCE:
            print(f"\n[EARLY STOPPING] No improvement for {PATIENCE} epochs")
            break
    
    print()

print("=" * 80)
print("FINE-TUNING COMPLETE")
print("=" * 80)
print(f"Best validation PR-AUC: {best_pr_auc:.4f}")
print(f"Model saved to: {checkpoint_save_path}")
print("=" * 80)

## 9. Visualize Training History

In [ ]:
import matplotlib.pyplot as plt

history_df = pd.DataFrame(train_history)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history_df['epoch'], history_df['train_loss'], label='Train', marker='o')
axes[0].plot(history_df['epoch'], history_df['val_loss'], label='Validation', marker='o')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# PR-AUC
axes[1].plot(history_df['epoch'], history_df['train_pr_auc'], label='Train', marker='o')
axes[1].plot(history_df['epoch'], history_df['val_pr_auc'], label='Validation', marker='o')
axes[1].axhline(y=zeroshot_image_metrics['pr_auc'], color='r', linestyle='--', label='Zero-shot')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('PR-AUC')
axes[1].set_title('PR-AUC')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# AUROC
axes[2].plot(history_df['epoch'], history_df['train_auroc'], label='Train', marker='o')
axes[2].plot(history_df['epoch'], history_df['val_auroc'], label='Validation', marker='o')
axes[2].axhline(y=zeroshot_image_metrics['auroc'], color='r', linestyle='--', label='Zero-shot')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('AUROC')
axes[2].set_title('AUROC')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Final Evaluation

Evaluate the fine-tuned model on the validation set.

In [ ]:
# Load best fine-tuned model
print("=" * 80)
print("LOADING BEST FINE-TUNED MODEL")
print("=" * 80)

best_checkpoint = torch.load(checkpoint_save_path, map_location=device, weights_only=False)
model.load_state_dict(best_checkpoint['model_state_dict'])
model.eval()

print(f"Loaded checkpoint from epoch {best_checkpoint['epoch']}")
print(f"Best validation PR-AUC: {best_checkpoint['best_pr_auc']:.4f}")
print("=" * 80)

In [ ]:
print("=" * 80)
print("FINAL EVALUATION (FINE-TUNED MODEL)")
print("=" * 80)

# Run inference on validation set
all_preds_ft = []
all_labels_ft = []
all_image_ids_ft = []

with torch.no_grad():
    for images, labels, image_ids in finetune_val_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        probs = model(images)
        
        all_preds_ft.extend(probs.cpu().numpy().tolist())
        all_labels_ft.extend(labels.cpu().numpy().tolist())
        all_image_ids_ft.extend(image_ids)

# Compute image-level metrics
finetuned_image_metrics = compute_metrics(
    np.array(all_preds_ft),
    np.array(all_labels_ft)
)

# Compute breast-level metrics
image_predictions_ft = {img_id: pred for img_id, pred in zip(all_image_ids_ft, all_preds_ft)}
breast_preds_ft, breast_labels_ft = aggregate_to_breast_level(
    image_predictions=image_predictions_ft,
    metadata=inbreast_val
)
finetuned_breast_metrics = compute_metrics(breast_preds_ft, breast_labels_ft)

print(f"\nValidation set: {len(all_preds_ft)} images, {len(breast_preds_ft)} breasts")
print()
print("Image-level metrics:")
print(f"  PR-AUC:  {finetuned_image_metrics['pr_auc']:.4f}")
print(f"  AUROC:   {finetuned_image_metrics['auroc']:.4f}")
print(f"  Brier:   {finetuned_image_metrics['brier']:.4f}")
print()
print("Breast-level metrics:")
print(f"  PR-AUC:  {finetuned_breast_metrics['pr_auc']:.4f}")
print(f"  AUROC:   {finetuned_breast_metrics['auroc']:.4f}")
print(f"  Brier:   {finetuned_breast_metrics['brier']:.4f}")
print("=" * 80)

## 11. Compare Zero-Shot vs Fine-Tuned

Analyze the improvement from fine-tuning.

In [ ]:
print("=" * 80)
print("TRANSFER LEARNING COMPARISON")
print("=" * 80)
print()
print(f"Model: Solution {solution_id}")
print()
print("Source Dataset (VinDr-Mammo):")
print(f"  PR-AUC: {selected_solution['pr_auc']:.4f}")
print(f"  AUROC:  {selected_solution['auroc']:.4f}")
print(f"  Brier:  {selected_solution['brier']:.4f}")
print()
print("Target Dataset (INbreast) - Zero-Shot:")
print(f"  Image PR-AUC:  {zeroshot_image_metrics['pr_auc']:.4f}")
print(f"  Image AUROC:   {zeroshot_image_metrics['auroc']:.4f}")
print(f"  Breast PR-AUC: {zeroshot_breast_metrics['pr_auc']:.4f}")
print(f"  Breast AUROC:  {zeroshot_breast_metrics['auroc']:.4f}")
print()
print("Target Dataset (INbreast) - Fine-Tuned:")
print(f"  Image PR-AUC:  {finetuned_image_metrics['pr_auc']:.4f} ({(finetuned_image_metrics['pr_auc'] - zeroshot_image_metrics['pr_auc']):.4f} improvement)")
print(f"  Image AUROC:   {finetuned_image_metrics['auroc']:.4f} ({(finetuned_image_metrics['auroc'] - zeroshot_image_metrics['auroc']):.4f} improvement)")
print(f"  Breast PR-AUC: {finetuned_breast_metrics['pr_auc']:.4f} ({(finetuned_breast_metrics['pr_auc'] - zeroshot_breast_metrics['pr_auc']):.4f} improvement)")
print(f"  Breast AUROC:  {finetuned_breast_metrics['auroc']:.4f} ({(finetuned_breast_metrics['auroc'] - zeroshot_breast_metrics['auroc']):.4f} improvement)")
print()

# Calculate improvement percentages
pr_auc_improvement = ((finetuned_breast_metrics['pr_auc'] - zeroshot_breast_metrics['pr_auc']) / zeroshot_breast_metrics['pr_auc']) * 100
auroc_improvement = ((finetuned_breast_metrics['auroc'] - zeroshot_breast_metrics['auroc']) / zeroshot_breast_metrics['auroc']) * 100

print("Relative Improvement (Breast-level):")
print(f"  PR-AUC: {pr_auc_improvement:+.2f}%")
print(f"  AUROC:  {auroc_improvement:+.2f}%")
print()

if pr_auc_improvement > 10:
    print("✓ SIGNIFICANT IMPROVEMENT: Fine-tuning provided major performance gains")
elif pr_auc_improvement > 5:
    print("✓ MODERATE IMPROVEMENT: Fine-tuning helped adapt to target dataset")
elif pr_auc_improvement > 0:
    print("✓ SMALL IMPROVEMENT: Minor gains from fine-tuning")
else:
    print("⚠ NO IMPROVEMENT: Zero-shot performance was comparable or better")

print("=" * 80)

## 12. Save Results

Save all results and fine-tuned model.

In [ ]:
from datetime import datetime
import json

print("=" * 80)
print("SAVING RESULTS")
print("=" * 80)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Prepare results dictionary
results = {
    "timestamp": timestamp,
    "solution_id": int(solution_id),
    "hyperparameters": {
        "learning_rate": float(selected_solution['learning_rate']),
        "weight_decay": float(selected_solution['weight_decay']),
        "dropout_rate": float(selected_solution['dropout_rate']),
        "augmentation_strength": float(selected_solution['augmentation_strength']),
        "unfreeze_fraction": float(selected_solution['unfreeze_fraction']),
    },
    "finetuning_config": {
        "epochs": FINETUNE_EPOCHS,
        "learning_rate": FINETUNE_LR,
        "weight_decay": FINETUNE_WEIGHT_DECAY,
        "augmentation_strength": FINETUNE_AUG_STRENGTH,
        "patience": PATIENCE,
    },
    "source_performance": {
        "dataset": "VinDr-Mammo",
        "pr_auc": float(selected_solution['pr_auc']),
        "auroc": float(selected_solution['auroc']),
        "brier": float(selected_solution['brier']),
    },
    "target_performance": {
        "dataset": "INbreast",
        "zero_shot": {
            "image_level": {
                "pr_auc": float(zeroshot_image_metrics['pr_auc']),
                "auroc": float(zeroshot_image_metrics['auroc']),
                "brier": float(zeroshot_image_metrics['brier']),
            },
            "breast_level": {
                "pr_auc": float(zeroshot_breast_metrics['pr_auc']),
                "auroc": float(zeroshot_breast_metrics['auroc']),
                "brier": float(zeroshot_breast_metrics['brier']),
            },
        },
        "finetuned": {
            "image_level": {
                "pr_auc": float(finetuned_image_metrics['pr_auc']),
                "auroc": float(finetuned_image_metrics['auroc']),
                "brier": float(finetuned_image_metrics['brier']),
            },
            "breast_level": {
                "pr_auc": float(finetuned_breast_metrics['pr_auc']),
                "auroc": float(finetuned_breast_metrics['auroc']),
                "brier": float(finetuned_breast_metrics['brier']),
            },
        },
    },
    "training_history": train_history,
}

# Save JSON
json_path = Path(FINETUNE_DIR) / f"finetune_results_solution_{solution_id}_{timestamp}.json"
with open(json_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f"[OK] Saved results: {json_path}")

# Save predictions CSV
predictions_df = pd.DataFrame({
    'image_id': all_image_ids_ft,
    'prediction_zeroshot': all_preds,
    'prediction_finetuned': all_preds_ft,
    'label': all_labels_ft,
})
csv_path = Path(FINETUNE_DIR) / f"predictions_solution_{solution_id}_{timestamp}.csv"
predictions_df.to_csv(csv_path, index=False)
print(f"[OK] Saved predictions: {csv_path}")

print()
print(f"Model saved to: {checkpoint_save_path}")
print(f"Results saved to: {FINETUNE_DIR}")
print("=" * 80)

## Summary

Fine-tuning complete! 

**Key Results:**
- Zero-shot performance provides baseline transfer capability
- Fine-tuning adapts the model to INbreast-specific patterns
- Both models and results saved to Google Drive

**Next Steps:**
- Fine-tune other Pareto solutions for comparison
- Experiment with different fine-tuning learning rates
- Analyze which hyperparameters transfer best
- Consider ensemble methods combining multiple fine-tuned models

**Saved Files:**
- Fine-tuned model checkpoint
- JSON results with all metrics
- CSV predictions (zero-shot vs fine-tuned)

Access at: `/content/drive/MyDrive/vindr_optimization/inbreast_finetuned/`